# 🤖 파이썬으로 웹 브라우저 조종하기! (Selenium 기초)

안녕하세요! 파이썬 챗봇 & 자동화 매니저입니다. 

지난번 `Beautiful Soup`이 웹사이트의 '정적인 HTML'을 가져오는 스크레이퍼였다면, 오늘 배울 `Selenium`(셀레니움)은 **실제 웹 브라우저(Firefox, Chrome 등)를 직접 조종하는** 강력한 자동화 도구입니다. 🕵️

요즘 웹사이트는 JavaScript(자바스크립트)를 사용해서 버튼을 누르거나 스크롤을 내릴 때마다 '동적으로' 내용이 바뀌죠? Selenium은 바로 이런 **동적인 웹사이트와 상호작용**하는 데 특화되어 있습니다.

이 튜토리얼을 통해 우리는 'Bandcamp'라는 음악 사이트를 자동으로 제어해서, 터미널에서 명령을 내리는 나만의 '음악 플레이어 봇'을 만들어 볼 거예요! 🎶

## 🔧 0단계: Colab 환경에 Selenium과 Firefox 설치하기

Google Colab은 기본적으로 웹 브라우저가 설치되어 있지 않은 리눅스 서버 환경입니다. 

따라서 Selenium을 사용하려면 3가지를 설치해야 합니다.

1.  `selenium` : 파이썬 라이브러리 (pip)
2.  `firefox` : 실제 조종할 웹 브라우저 (apt-get)
3.  `geckodriver` : 파이썬과 Firefox를 연결해주는 '번역기' (wget)

In [ ]:
# 1. 파이썬용 selenium 라이브러리 설치
!pip install selenium

# 2. Colab에 Firefox 브라우저 설치
!apt-get update
!apt-get install firefox

# 3. geckodriver (Firefox 조종 드라이버) 다운로드 및 설치
# (특정 버전을 다운로드합니다. Colab 환경에 따라 버전이 달라질 수 있습니다.)
!wget https://github.com/mozilla/geckodriver/releases/download/v0.34.0/geckodriver-v0.34.0-linux64.tar.gz
!tar -xvzf geckodriver-v0.34.0-linux64.tar.gz
!chmod +x geckodriver
!mv geckodriver /usr/local/bin/ # 시스템 PATH에 추가

print("🎉 설치 완료! 이제 Selenium으로 Firefox를 조종할 수 있습니다.")

## 👻 1단계: '유령 브라우저' 켜고 접속하기 (Headless Mode)

Selenium의 핵심은 `webdriver`입니다. `webdriver.Firefox()`를 호출하면 진짜 Firefox 창이 뿅! 하고 뜹니다.

하지만 Colab 같은 서버 환경에서는 눈에 보이는 창이 필요 없겠죠? 이때 **'Headless 모드'**를 사용합니다. `Options` 객체에 `"-headless"` 인자를 추가하면, 브라우저가 화면 없이 '유령'처럼 백그라운드에서 실행됩니다.

* `driver = webdriver.Firefox(options=options)` : 헤드리스 브라우저 실행
* `driver.get(URL)` : 원하는 주소(URL)로 접속
* `driver.title` : 현재 페이지의 '제목' 가져오기
* `driver.quit()` : **(아주 중요!)** 브라우저를 완전히 종료합니다. 이걸 안 하면 '좀비 브라우저'가 메모리를 계속 차지해요! 🧟

In [ ]:
from selenium import webdriver
from selenium.webdriver.firefox.options import Options

# 헤드리스 모드 설정을 위한 'Options' 객체 생성
options = Options()
options.add_argument("-headless")

# 'options'를 적용하여 Firefox 웹 드라이버(브라우저)를 시작합니다.
driver = webdriver.Firefox(options=options)

print("유령 브라우저가 켜졌습니다...")

# python.org 사이트로 접속!
driver.get("https://www.python.org")

# 'driver.title'로 현재 페이지의 제목을 가져올 수 있습니다.
print(f"방금 접속한 페이지 제목: {driver.title}")

# (필수!) 작업이 끝나면 브라우저를 종료합니다.
driver.quit()
print("브라우저가 종료되었습니다.")

## 🎯 2단계: 원하는 정보 '콕 집어' 찾기 (Locators)

브라우저가 페이지에 접속했다면, 이제 `Beautiful Soup`처럼 원하는 요소를 찾아야 합니다. 

Selenium의 최신 방식은 `By` 객체를 사용하는 것입니다. `Beautiful Soup`의 `find(id=...)`, `find_all(class_=...)`과 비슷해요.

* `driver.find_element(By.ID, "my-id")` : ID로 1개 찾기
* `driver.find_elements(By.CLASS_NAME, "my-class")` : 클래스로 '모두' 찾기 (리스트 반환)
* `driver.find_element(By.CSS_SELECTOR, "div.meta a strong")` : CSS 선택자로 1개 찾기 (강력 추천!)

우리의 목표 사이트인 `https://bandcamp.com/discover/`에 접속해서 정보를 찾아봅시다.

In [ ]:
from selenium.webdriver.common.by import By

options = Options()
options.add_argument("-headless")
driver = webdriver.Firefox(options=options)

print("Bandcamp Discover 페이지로 접속합니다...")
driver.get("https://bandcamp.com/discover/")

# 1. ID로 찾기: 'view-more' (더 보기) 버튼 찾기
# (개발자 도구(F12)로 확인해보면 id가 'view-more'입니다.)
pagination_button = driver.find_element(By.ID, "view-more")
print(f"찾은 버튼 텍스트: {pagination_button.accessible_name}")

# 2. CLASS_NAME으로 찾기: 모든 '트랙 아이템' 찾기
tracks = driver.find_elements(By.CLASS_NAME, "results-grid-item")
print(f"현재 페이지에 로드된 트랙 수: {len(tracks)}")

# 3. CSS_SELECTOR로 찾기: 첫 번째 트랙(tracks[0]) 안에서 '앨범명' 찾기
# (CSS 선택자: 'div.meta' 클래스 안의 'a' 태그 안의 'strong' 태그)
album_title = tracks[0].find_element(By.CSS_SELECTOR, "div.meta a strong")
print(f"첫 번째 트랙의 앨범명: {album_title.text}")

driver.quit()

## ⌨️ 3단계: 요소와 '상호작용'하기 (Click, Send Keys)

Selenium의 진짜 힘은 요소를 '클릭'하거나 '키보드 입력'을 할 수 있다는 것입니다.

* `element.click()` : 찾은 요소를 마우스로 클릭합니다. (버튼, 링크 등)
* `element.send_keys("검색어")` : 찾은 요소(검색창)에 텍스트를 입력합니다.
* `element.submit()` : 폼(form)을 '제출'합니다. (Enter 키 누르기)

**[중요!]** 요즘 웹사이트는 '쿠키 동의' 팝업이 뜨는 경우가 많죠? 자동화 봇도 이 팝업을 '클릭'해서 닫아줘야 다음 단계로 진행할 수 있습니다. 팝업이 없을 수도 있으니 `try...except`로 감싸주는 것이 안전합니다.

In [ ]:
from selenium.common.exceptions import NoSuchElementException
import time

options = Options()
options.add_argument("-headless")
driver = webdriver.Firefox(options=options)

driver.get("https://bandcamp.com/discover/")
print("Bandcamp 접속...")

# 1. [상호작용] 쿠키 동의 팝업창 처리 (있을 수도, 없을 수도 있음)
try:
  # CSS 선택자로 'Accept necessary only' 버튼 찾기
  cookie_btn = driver.find_element(By.CSS_SELECTOR, "#cookie-control-dialog button.g-button.outline")
  cookie_btn.click() # 버튼 클릭!
  print("쿠키 동의 버튼을 클릭했습니다.")
except NoSuchElementException: # 요소를 못 찾으면 (팝업이 없으면)
  print("쿠키 동의 팝업이 없네요. 통과!")

# 2. [상호작용] 검색창에 'selenium' 입력하고 검색하기
print("검색창에 'selenium' 입력...")
search_field = driver.find_element(By.TAG_NAME, "input") # <input> 태그 찾기
search_field.send_keys("selenium") # 'selenium' 텍스트 입력
search_field.submit() # Enter 키 눌러서 제출

time.sleep(2) # 검색 결과 로딩을 위해 2초만 기다리기 (좋은 방법은 아님!)

print(f"검색 결과 페이지 제목: {driver.title}")

driver.quit()

## ⏱️ 4단계: '기다림'의 미학 (Implicit & Explicit Waits)

방금 `time.sleep(2)`를 썼죠? 이건 아주 나쁜 방법입니다. ❌
인터넷이 느리면 2초 안에 로딩이 안 될 수도 있고, 빠르면 0.1초 만에 되는데 1.9초를 낭비하니까요.

Selenium은 똑똑하게 '기다리는' 2가지 방법을 제공합니다.

1.  **Implicit Wait (암시적 대기)**
    * `driver.implicitly_wait(5)`
    * '전역' 설정. 
    * 앞으로 어떤 요소를 찾든, 만약 바로 안 보이면 최대 5초까지는 기다려줘.
    * 딱 한 번만 설정하면 됩니다. 웬만하면 켜두는 게 좋아요.

2.  **Explicit Wait (명시적 대기)**
    * `WebDriverWait(driver, 10).until(EC. ... )`
    * '특정 조건'을 설정. 
    * 최대 10초까지 기다리는데, 저 버튼이 '클릭 가능'해질 때까지 기다려줘.
    * JavaScript 로딩이 끝나는 시점을 정확히 포착할 때 (예: '더 보기' 버튼) 필수입니다.

In [ ]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

options = Options()
options.add_argument("-headless")
driver = webdriver.Firefox(options=options)

# 1. Implicit Wait 설정: 앞으로 요소를 찾을 때 최대 5초까지 기다림
driver.implicitly_wait(5)

driver.get("https://bandcamp.com/discover/")
print("Bandcamp 접속...")

# 쿠키 팝업 처리 (try...except...)
try:
  driver.find_element(By.CSS_SELECTOR, "#cookie-control-dialog button.g-button.outline").click()
  print("쿠키 동의 완료.")
except NoSuchElementException:
  print("쿠키 팝업 없음.")

# --- '더 보기' 버튼 클릭 테스트 --- #
tracks_before = len(driver.find_elements(By.CLASS_NAME, "results-grid-item"))
print(f"'더 보기' 클릭 전 트랙 수: {tracks_before}")

# 'view-more' 버튼 클릭
pagination_button = driver.find_element(By.ID, "view-more")
pagination_button.click()

# 2. Explicit Wait 설정: (중요!)
# '더 보기'를 누르면 버튼이 잠시 비활성화됐다가 로딩이 끝나면 다시 활성화됩니다.
# 'view-more' 버튼이 '다시 클릭 가능(clickable)해질 때까지' 최대 10초간 기다립니다.
wait = WebDriverWait(driver, 10) # 10초 대기 객체 생성
wait.until(EC.element_to_be_clickable((By.ID, "view-more")))

print("추가 트랙 로딩 완료! (Explicit Wait 성공)")

tracks_after = len(driver.find_elements(By.CLASS_NAME, "results-grid-item"))
print(f"'더 보기' 클릭 후 트랙 수: {tracks_after}")

driver.quit()

## 🏛️ 5단계: 똑똑하게 코드 관리하기 (Page Object Model)

축하합니다! Selenium의 핵심 기능(찾기, 클릭, 대기)을 모두 배웠습니다.

하지만 코드가 길어지면 `driver.find_element(By.ID, "view-more")` 같은 '찾기' 코드가 여기저기 흩어져서 지저분해집니다. 만약 Bandcamp가 웹사이트 디자인을 바꿔서 `id`가 `"view-more"`에서 `"load-more"`로 바뀌면? 우리 코드 100군데를 다 고쳐야 할까요? 😱

이걸 방지하는 디자인 패턴이 **Page Object Model (POM)**입니다.

규칙은 간단합니다.
1.  **Locators (로케이터)**: `(By.ID, "view-more")` 같은 '주소' 정보만 따로 모아 클래스로 관리합니다. (`locators.py`)
2.  **Elements (요소)**: '트랙 1개'처럼 반복되는 요소의 '동작'(재생, 정지)을 클래스로 묶습니다. (`elements.py`)
3.  **Pages (페이지)**: 'Discover 페이지' 전체의 '동작'(쿠키 닫기, 트랙 목록 가져오기)을 클래스로 묶습니다. (`pages.py`)

원래는 `.py` 파일을 분리하지만, Colab에서는 이 모든 클래스를 아래 셀들에 순서대로 정의해 보겠습니다.

### 5-1. Locators 정의하기
정보가 바뀔 가능성이 가장 높은 '주소'들만 모아둡니다.

In [ ]:
# (원래는 bandcamp/web/locators.py 파일)
from selenium.webdriver.common.by import By

class DiscoverPageLocator:
  # (By.CLASS_NAME, "results-grid")
  DISCOVER_RESULTS = (By.CLASS_NAME, "results-grid") 
  # ("#cookie-control-dialog button.g-button.outline")
  COOKIE_ACCEPT_NECESSARY = (
      By.CSS_SELECTOR, 
      "#cookie-control-dialog button.g-button.outline"
  )

class TrackListLocator:
  ITEM = (By.CLASS_NAME, "results-grid-item")
  PAGINATION_BUTTON = (By.ID, "view-more")

class TrackLocator:
  PLAY_BUTTON = (By.CSS_SELECTOR, "button.play-pause-button")
  URL = (By.CSS_SELECTOR, "div.meta p a")
  ALBUM = (By.CSS_SELECTOR, "div.meta p a strong")
  ARTIST = (By.CSS_SELECTOR, "div.meta p a span")
  GENRE = (By.CSS_SELECTOR, "div.meta p.genre")

print("로케이터(주소) 클래스 정의 완료!")

### 5-2. Base 및 Elements 정의하기
`driver`와 `wait` 객체를 물려주는 '부모' 클래스(`WebPage`, `WebComponent`)와, '트랙 1개'의 동작(`TrackElement`), '트랙 목록'의 동작(`TrackListElement`)을 정의합니다.

In [ ]:
# (원래는 bandcamp/web/base.py 와 elements.py 파일)
from selenium.webdriver.remote.webdriver import WebDriver
from selenium.webdriver.remote.webelement import WebElement
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException
from dataclasses import dataclass
from pprint import pformat

MAX_WAIT_SECONDS = 10.0
DEFAULT_WINDOW_SIZE = (1920, 3000) # 창 크기 고정 (더 많은 트랙 로드)

# 트랙 정보를 저장할 데이터 클래스
@dataclass
class Track:
  album: str
  artist: str
  genre: str
  url: str
  def __str__(self):
    return pformat(self) # 예쁘게 출력하기

# 모든 '페이지'의 부모가 될 클래스
class WebPage:
  def __init__(self, driver: WebDriver) -> None:
    self._driver = driver
    self._driver.set_window_size(*DEFAULT_WINDOW_SIZE)
    self._driver.implicitly_wait(5)
    self._wait = WebDriverWait(driver, MAX_WAIT_SECONDS)

# 모든 '요소'의 부모가 될 클래스
class WebComponent(WebPage):
  def __init__(self, parent: WebElement, driver: WebDriver) -> None:
    super().__init__(driver)
    self._parent = parent # 부모 요소 (예: 트랙 카드 <div>)

# --- 진짜 '요소' 클래스 --- #

class TrackElement(WebComponent):
  """트랙 1개의 동작 (재생, 정지, 정보 가져오기)"""
  def play(self) -> None:
    if not self.is_playing:
      self._get_play_button().click()
  
  def pause(self) -> None:
    if self.is_playing:
      self._get_play_button().click()

  @property
  def is_playing(self) -> bool:
    # 'aria-label' 속성이 'Pause'를 포함하면 재생 중
    return "Pause" in self._get_play_button().get_attribute("aria-label")

  def _get_play_button(self):
    # 부모(트랙 카드) 안에서 PLAY_BUTTON을 찾음
    return self._parent.find_element(*TrackLocator.PLAY_BUTTON)

  def get_track_info(self) -> Track:
    """트랙 정보(앨범, 아티스트 등)를 Track 객체로 반환"""
    full_url = self._parent.find_element(*TrackLocator.URL).get_attribute("href")
    clean_url = full_url.split("?")[0] if full_url else ""
    try:
      genre = self._parent.find_element(*TrackLocator.GENRE).text
    except NoSuchElementException: # 장르가 없는 트랙도 있음
      genre = ""
    
    return Track(
        album=self._parent.find_element(*TrackLocator.ALBUM).text,
        artist=self._parent.find_element(*TrackLocator.ARTIST).text,
        genre=genre,
        url=clean_url,
    )

class TrackListElement(WebComponent):
  """트랙 '목록'의 동작 (트랙 가져오기, 더 보기 클릭)"""
  def __init__(self, parent: WebElement, driver: WebDriver) -> None:
    super().__init__(parent, driver)
    self.available_tracks = self._get_available_tracks()
  
  def load_more(self) -> None:
    """'더 보기' 버튼을 누르고 새 트랙 목록을 갱신"""
    view_more_button = self._driver.find_element(*TrackListLocator.PAGINATION_BUTTON)
    view_more_button.click()
    # 로딩이 끝날 때까지 (버튼이 다시 클릭 가능해질 때까지) 기다림
    self._wait.until(
        EC.element_to_be_clickable(TrackListLocator.PAGINATION_BUTTON)
    )
    self.available_tracks = self._get_available_tracks() # 트랙 목록 갱신

  def _get_available_tracks(self) -> list[TrackElement]:
    """현재 보이는 모든 트랙 요소를 TrackElement 리스트로 반환"""
    all_tracks = self._driver.find_elements(*TrackListLocator.ITEM)
    return [
        TrackElement(track, self._driver)
        for track in all_tracks
        if track.is_displayed() and track.text.strip() # 보이는 트랙만 필터링
    ]

print("Base 및 Elements 클래스 정의 완료!")

### 5-3. Page 정의하기
'Discover 페이지' 전체를 나타내는 클래스입니다. 이 클래스는 '트랙 목록'(`TrackListElement`)을 포함합니다.

In [ ]:
# (원래는 bandcamp/web/pages.py 파일)

class DiscoverPage(WebPage):
  """Bandcamp Discover 페이지 모델"""
  def __init__(self, driver: WebDriver) -> None:
    super().__init__(driver)
    self._accept_cookie_consent() # 접속하자마자 쿠키 팝업 닫기
    self.discover_tracklist = TrackListElement(
        self._driver.find_element(*DiscoverPageLocator.DISCOVER_RESULTS),
        self._driver,
    )

  def _accept_cookie_consent(self) -> None:
    """쿠키 동의 버튼 (있으면) 누르기"""
    try:
      self._driver.find_element(*DiscoverPageLocator.COOKIE_ACCEPT_NECESSARY).click()
    except NoSuchElementException:
      pass # 팝업 없으면 통과

print("Page 클래스 정의 완료!")

## 🎶 6단계: 음악 플레이어 '봇' 만들기 (Player & TUI)

이제 모든 '부품(POM 클래스)'이 준비되었습니다. 

이 부품들을 조립해서 실제 '음악 플레이어 봇'의 로직을 만듭니다.

1.  `Player` 클래스: 브라우저를 켜고 끄며, `DiscoverPage`를 제어하는 '컨트롤 타워'입니다.
2.  TUI 함수 (`display_tracks` 등): 사용자에게 트랙 목록을 '보여주고' 명령을 받는 '인터페이스'입니다.

In [ ]:
# (원래는 bandcamp/app/player.py 와 tui.py 파일)
import time

BANDCAMP_DISCOVER_URL = "https://bandcamp.com/discover/"

class Player:
  """플레이어 컨트롤 타워. 브라우저를 켜고 끔."""
  def __init__(self) -> None:
    self._driver = self._set_up_driver()
    self.page = DiscoverPage(self._driver)
    self.tracklist = self.page.discover_tracklist
    self._current_track = self.tracklist.available_tracks[0]

  def __enter__(self): # with Player() as player: 구문 지원
    return self
  
  def __exit__(self, exc_type, exc_value, exc_tb): # with 구문이 끝날 때
    print("\n음악 플레이어 봇 종료. (브라우저 닫는 중)")
    self._driver.quit() # 브라우저를 '반드시' 닫아줌
  
  def play(self, track_number=None):
    """트랙 번호로 재생 (번호가 없으면 현재 트랙 재생)"""
    if track_number:
      if 1 <= track_number <= len(self.tracklist.available_tracks):
        self._current_track = self.tracklist.available_tracks[track_number - 1]
      else:
        raise IndexError("유효하지 않은 트랙 번호입니다.")
    self._current_track.play()

  def pause(self):
    self._current_track.pause()

  def _set_up_driver(self):
    """헤드리스 브라우저 켜고 Bandcamp 접속"""
    options = Options()
    options.add_argument("-headless")
    browser = webdriver.Firefox(options=options)
    browser.get(BANDCAMP_DISCOVER_URL)
    return browser

# --- TUI (Text-based User Interface) 함수 --- #

COLUMN_WIDTH = CW = 30 # 컬럼 너비

def display_tracks(player):
  """트랙 목록을 예쁘게 표로 출력"""
  header = f"{'#':<5} {'Album':<{CW}} {'Artist':<{CW}} {'Genre':<{CW}}"
  print(header)
  print("-" * (5 + CW * 3))
  
  for i, track_element in enumerate(player.tracklist.available_tracks, start=1):
    track = track_element.get_track_info()
    # 텍스트가 너무 길면 '...'으로 자르기
    album = (track.album[:CW-3] + "...") if len(track.album) > CW else track.album
    artist = (track.artist[:CW-3] + "...") if len(track.artist) > CW else track.artist
    genre = (track.genre[:CW-3] + "...") if len(track.genre) > CW else track.genre
    
    print(f"{i:<5} {album:<{CW}} {artist:<{CW}} {genre:<{CW}}")

def play_track(player, track_number=None):
  """트랙 재생 TUI 함수"""
  try:
    player.play(track_number)
    print("\n▶️ 지금 재생 중인 곡:")
    print(player._current_track.get_track_info())
    print("\n(주의: Colab 환경에서는 실제로 소리가 들리지 않습니다!)")
  except IndexError as e:
    print(f"오류: {e}")

def pause_track(player):
  """트랙 정지 TUI 함수"""
  player.pause()
  print("\n⏸️ 재생을 멈췄습니다.")

print("Player 및 TUI 함수 정의 완료!")

## 🚀 7단계: 최종 실행!

이제 모든 부품이 조립되었습니다! `with Player() as player:` 구문을 사용해서 봇을 안전하게 실행해 봅시다.

TUI(텍스트 인터페이스) 함수들을 호출해서 봇을 제어하는 시뮬레이션을 해보겠습니다.

In [ ]:
print("음악 플레이어 봇을 시작합니다... (헤드리스 브라우저 켜는 중)")

# with Player() as player: 로 봇을 실행합니다.
# 이 블록이 끝나면 __exit__ 메서드가 자동 호출되어 브라우저가 종료됩니다.
with Player() as player:
  print("플레이어 준비 완료! (Bandcamp 접속 및 쿠키 처리 완료)")
  
  # 1. 현재 트랙 목록 보여주기
  print("\n--- 1. 'tracks' 명령: 현재 트랙 목록 --- ")
  display_tracks(player)
  
  # 2. 3번 트랙 재생하기
  print("\n--- 2. 'play 3' 명령: 3번 트랙 재생 --- ")
  play_track(player, 3)
  time.sleep(2) # 2초간 '듣는' 척하기

  # 3. 재생 정지하기
  print("\n--- 3. 'pause' 명령: 재생 정지 --- ")
  pause_track(player)

  # 4. 'more' 버튼 눌러서 트랙 더 가져오기
  print("\n--- 4. 'more' 명령: 트랙 더 가져오기 (로딩 대기 중...) --- ")
  player.tracklist.load_more()
  print("트랙 추가 로드 완료!")
  display_tracks(player) # 늘어난 목록 다시 표시
  
  # 5. 새로 로드된 12번 트랙 재생하기
  print("\n--- 5. 'play 12' 명령: 12번 트랙 재생 --- ")
  play_track(player, 12)
  time.sleep(2) 

# with 블록이 여기서 끝납니다. __exit__이 호출됩니다.

## 🎯 혼자 해보기: 실생활 연습 문제 10선 🎯

이제 여러분 차례입니다! Selenium은 'JavaScript로 동적으로 로드되는' 사이트에서 진가를 발휘합니다.

아래 사이트들은 JS 로딩이나 상호작용이 필요한 곳들입니다. 힌트 코드를 드릴 테니, `___` 부분만 채워 넣으면서 도전해보세요! 

**(주의: 아래 코드들은 독립적으로 실행해야 합니다. `driver`를 매번 새로 켜고 닫아주세요.)**

### 📜 연습 사이트 1: Quotes to Scrape (JavaScript 버전)
**URL:** `https://quotes.toscrape.com/js/`
(이 사이트는 JavaScript로 명언을 불러옵니다. `Beautiful Soup`으로는 첫 페이지만 보입니다.)

**문제 1:** 'Next' 버튼을 클릭해서 2페이지로 이동한 후, 2페이지의 '첫 번째' 명언 텍스트 가져오기
**힌트:** 'Next' 버튼은 `li.next a` CSS 선택자로 찾을 수 있습니다. 2페이지 로딩을 위해 '명시적 대기'를 사용해야 합니다. (1페이지의 첫 번째 명언과 텍스트가 '달라질' 때까지 기다려보세요!)

In [ ]:
options = Options()
options.add_argument("-headless")
driver = webdriver.Firefox(options=options)
driver.implicitly_wait(5)
wait = WebDriverWait(driver, 10)

url = "https://quotes.toscrape.com/js/"
driver.get(url)

# 1페이지의 첫 번째 명언 텍스트를 미리 저장
first_quote_text = driver.find_element(By.CSS_SELECTOR, "span.text").text
print(f"1페이지 첫 명언: {first_quote_text}")

# 'Next' 버튼을 찾아서 클릭
next_button = driver.find_element(By.CSS_SELECTOR, "li.next a")
next_button.click()

# [중요] 2페이지 로딩 대기!
# 1페이지의 첫 번째 명언이 '사라질 때까지(staleness_of)' 기다립니다.
wait.until(EC.staleness_of(driver.find_element(By.CSS_SELECTOR, "span.text")))
# (다른 방법: 2페이지의 'span.text'가 *나타날 때까지* 기다리기)
# wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "span.text")))

# 2페이지의 첫 번째 명언 텍스트 가져오기
second_page_quote = driver.find_element(By.CSS_SELECTOR, "span.text").text
print(f"2페이지 첫 명언: {second_page_quote}")

driver.quit()

**문제 2:** 'Next' 버튼을 '2번' 클릭해서 3페이지로 이동한 후, 3페이지의 '첫 번째' 명언의 '작가' 이름 가져오기

In [ ]:
options = Options()
options.add_argument("-headless")
driver = webdriver.Firefox(options=options)
driver.implicitly_wait(5)
wait = WebDriverWait(driver, 10)
url = "https://quotes.toscrape.com/js/"
driver.get(url)

# 1. 'Next' 버튼 클릭 (1 -> 2페이지)
driver.find_element(By.CSS_SELECTOR, "li.next a").click()
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "span.text"))) # 로딩 대기
print("2페이지 로드 완료.")

# 2. 'Next' 버튼 *또* 클릭 (2 -> 3페이지)
driver.find_element(By.CSS_SELECTOR, "___.next ___ ").click()
wait.until(EC.staleness_of(driver.find_element(By.CSS_SELECTOR, "span.text"))) # 2페이지 명언이 사라질 때까지 대기
print("3페이지 로드 완료.")

# 3. 3페이지의 첫 번째 '작가' 이름 가져오기
author = driver.find_element(By.CSS_SELECTOR, "small.___").text
print(f"3페이지 첫 작가: {author}")

driver.quit()

### 🖥️ 연습 사이트 2: WhatIsMyBrowser.com
**URL:** `https://www.whatismybrowser.com/`
(이 사이트는 접속한 브라우저의 정보를 JavaScript로 분석해서 보여줍니다.)

**문제 3:** Selenium이 켠 Firefox 브라우저의 '이름'과 '버전' 알아내기
**힌트:** 브라우저 이름은 `string-major`라는 `class`를 가진 `a` 태그에, 버전은 `string-major` 바로 다음 형제(sibling) `span` 태그에 있습니다.

In [ ]:
options = Options()
options.add_argument("-headless")
driver = webdriver.Firefox(options=options)
driver.implicitly_wait(5)
url = "https://www.whatismybrowser.com/"
driver.get(url)

# 브라우저 이름 찾기
browser_name = driver.find_element(By.CSS_SELECTOR, "a.string-major").text

# 브라우저 버전 찾기 (XPath 사용: 'a.string-major'의 'following-sibling::span' (다음 형제 span))
browser_version = driver.find_element(By.XPATH, "//a[@class='string-major']/following-sibling::span").text

print(f"Selenium이 켠 브라우저: {browser_name} / 버전: {browser_version}")

driver.quit()

### 🔒 연습 사이트 3: The Internet - 로그인 폼
**URL:** `https://the-internet.herokuapp.com/login`

**문제 4:** ID(`tomsmith`), PW(`SuperSecretPassword!`)를 입력하고 'Login' 버튼 클릭하기

In [ ]:
options = Options()
options.add_argument("-headless")
driver = webdriver.Firefox(options=options)
driver.implicitly_wait(5)
url = "https://the-internet.herokuapp.com/login"
driver.get(url)

# 1. ID 입력창(id='username')을 찾아 'tomsmith' 입력
driver.find_element(By.ID, "___").send_keys("tomsmith")

# 2. PW 입력창(id='password')을 찾아 'SuperSecretPassword!' 입력
driver.find_element(By.ID, "___").send_keys("SuperSecretPassword!")

# 3. Login 버튼(css='button[type="submit"]')을 찾아 클릭
driver.find_element(By.CSS_SELECTOR, "button[type='submit']").click()

# 4. 로그인 성공 메시지(css='div.flash.success')의 텍스트 가져오기
success_message = driver.find_element(By.CSS_SELECTOR, "div.flash.___").text

print(f"로그인 결과: {success_message.strip()}")

driver.quit()

**문제 5:** '틀린' ID(`wrong`), '틀린' PW(`wrong`)를 입력하고 로그인 시도 후, '실패' 메시지 가져오기
**힌트:** 실패 메시지는 `div.flash.error` CSS 선택자로 찾을 수 있습니다.

In [ ]:
options = Options()
options.add_argument("-headless")
driver = webdriver.Firefox(options=options)
driver.implicitly_wait(5)
url = "https://the-internet.herokuapp.com/login"
driver.get(url)

driver.find_element(By.ID, "username").send_keys("wrong")
driver.find_element(By.ID, "password").send_keys("wrong")
driver.find_element(By.CSS_SELECTOR, "button[type='submit']").click()

# 실패 메시지(css='div.flash.error')의 텍스트 가져오기
error_message = driver.find_element(By.CSS_SELECTOR, "div.flash.___").text

print(f"로그인 결과: {error_message.strip()}")

driver.quit()

### ⏳ 연습 사이트 4: The Internet - 동적 로딩 (Explicit Wait 연습)
**URL:** `https://the-internet.herokuapp.com/dynamic_loading/1`
(Start 버튼을 누르면 로딩 후 'Hello World!'가 나타납니다.)

**문제 6:** 'Start' 버튼을 누르고, 'Hello World!' 텍스트가 '보일 때까지' 기다렸다가 텍스트 가져오기

In [ ]:
options = Options()
options.add_argument("-headless")
driver = webdriver.Firefox(options=options)
wait = WebDriverWait(driver, 10)
url = "https://the-internet.herokuapp.com/dynamic_loading/1"
driver.get(url)

# 1. 'Start' 버튼(css='div#start button') 클릭
driver.find_element(By.CSS_SELECTOR, "div#start button").click()

# 2. 'Hello World!' 텍스트(id='finish')가 '보일 때까지(visibility_of_element_located)' 기다리기
wait.until(EC.visibility_of_element_located((By.ID, "___")))

# 3. 텍스트 가져오기
hello_text = driver.find_element(By.ID, "finish").text
print(f"로딩 완료! 나타난 텍스트: {hello_text}")

driver.quit()

### 🛒 연습 사이트 5: 네이버 쇼핑 (상호작용)
**URL:** `https://shopping.naver.com/`

**문제 7:** 네이버 쇼핑 검색창에 '셀레니움' 입력하고 검색한 후, 검색 결과 '첫 번째' 상품의 '제목' 가져오기

In [ ]:
options = Options()
options.add_argument("-headless")
driver = webdriver.Firefox(options=options)
driver.implicitly_wait(5)
wait = WebDriverWait(driver, 10)
url = "https://shopping.naver.com/"
driver.get(url)

# 1. 네이버 쇼핑 검색창(css='input._searchInput_search_input_QXUFf') 찾기
# (네이버는 class 이름이 복잡하고 자주 바뀝니다. 지금은 이걸로 동작합니다.)
search_bar = driver.find_element(By.CSS_SELECTOR, "input._searchInput_search_input_QXUFf")
search_bar.send_keys("셀레니움")
search_bar.submit()

# 2. 검색 결과 페이지 로딩 대기 (첫 번째 상품(css='div.product_title__Mmw2K')이 보일 때까지)
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.product_title__Mmw2K")))

# 3. 첫 번째 상품 제목 가져오기
first_item_title = driver.find_element(By.CSS_SELECTOR, "div.product_title__Mmw2K a").text
print(f"'셀레니움' 검색 결과 1위 상품: {first_item_title}")

driver.quit()

**문제 8:** '셀레니움' 검색 후, '낮은 가격순'으로 정렬하기
**힌트:** '낮은 가격순' 정렬 버튼은 `a` 태그이면서 `data-sort-key` 속성값이 `price_asc` 입니다. `(By.CSS_SELECTOR, "a[data-sort-key='price_asc']")`

In [ ]:
options = Options()
options.add_argument("-headless")
driver = webdriver.Firefox(options=options)
driver.implicitly_wait(5)
wait = WebDriverWait(driver, 10)
url = "https://shopping.naver.com/"
driver.get(url)

search_bar = driver.find_element(By.CSS_SELECTOR, "input._searchInput_search_input_QXUFf")
search_bar.send_keys("셀레니움")
search_bar.submit()
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.product_title__Mmw2K")))

# '낮은 가격순' 버튼 찾기
low_price_button = driver.find_element(By.CSS_SELECTOR, "a[data-sort-key='___']")
low_price_button.click()

# 정렬이 완료될 때까지 잠시 대기 (버튼이 다시 'selected' 상태가 될 때까지)
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "a[data-sort-key='price_asc'][class*='selected']")))
print("낮은 가격순 정렬 완료!")

# 정렬된 후 첫 번째 상품 제목 가져오기
first_item_title = driver.find_element(By.CSS_SELECTOR, "div.product_title__Mmw2K a").text
print(f"가장 저렴한 '셀레니움' 상품: {first_item_title}")

driver.quit()

### ✅ 연습 사이트 6: The Internet - 체크박스
**URL:** `https://the-internet.herokuapp.com/checkboxes`

**문제 9:** 'checkbox 1'은 '체크'하고, 'checkbox 2'는 '체크 해제'하기
**힌트:** `element.is_selected()`로 현재 체크 상태를 확인한 후, `.click()`으로 상태를 변경하세요. (CSS 선택자: `form#checkboxes input[type='checkbox']`)

In [ ]:
options = Options()
options.add_argument("-headless")
driver = webdriver.Firefox(options=options)
driver.implicitly_wait(5)
url = "https://the-internet.herokuapp.com/checkboxes"
driver.get(url)

# 모든 체크박스를 찾습니다.
checkboxes = driver.find_elements(By.CSS_SELECTOR, "form#checkboxes input[type='checkbox']")

checkbox1 = checkboxes[0]
checkbox2 = checkboxes[1]

# 문제: checkbox 1은 '체크' (만약 체크가 안 되어 있다면)
if not checkbox1.is_selected():
  checkbox1.click()
  print("체크박스 1을 클릭해서 '체크'했습니다.")

# 문제: checkbox 2는 '체크 해제' (만약 체크가 되어 있다면)
if checkbox2.is_selected():
  checkbox2.click()
  print("체크박스 2를 클릭해서 '해제'했습니다.")

print(f"최종 상태 - 1: {checkbox1.is_selected()}, 2: {checkbox2.is_selected()}")

driver.quit()

### 💯 연습 사이트 7: '가짜 파이썬 직업' (복습)
**URL:** `https://realpython.github.io/fake-jobs/`

**문제 10:** 'Senior'가 포함된 공고의 '제목'과 '회사명'만 모두 출력하기
**힌트:** `find_elements`로 모든 카드를 찾은(`div.card-content`) 다음, `for` 문으로 반복하면서 `h2.title` 텍스트에 'Senior'가 있는지 확인하세요.

In [ ]:
options = Options()
options.add_argument("-headless")
driver = webdriver.Firefox(options=options)
driver.implicitly_wait(5)
url = "https://realpython.github.io/fake-jobs/"
driver.get(url)

# 모든 '공고 카드'를 찾습니다.
job_cards = driver.find_elements(By.CLASS_NAME, "card-content")

print("--- 'Senior' 직무 공고 목록 ---")
for card in job_cards:
  # 카드 안에서 '제목'을 찾습니다.
  title_element = card.find_element(By.CSS_SELECTOR, "h2.title")
  title_text = title_element.text
  
  # 만약 '제목'에 'Senior'가 포함되어 있다면
  if "___" in title_text:
    # '회사명'도 찾아서 함께 출력합니다.
    company = card.find_element(By.CSS_SELECTOR, "h3.___").text
    print(f"[직무] {title_text} \n[회사] {company}\n")

driver.quit()